In [1]:
# Setup and imports
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter

from app.database import CourseDatabase
from app.course import Course
# Use the Course object's built-in visualization methods
# These use the same functions we defined above, but with production column names

# Reload course module to pick up new methods
import importlib
import app.course as course_module
importlib.reload(course_module)
from app.course import Course
# Create Course object with custom smoothing parameters

In [2]:
def plot_smoothing_comparison(df, title_prefix="", 
                               gradient_raw_col='raw_gradient', gradient_smooth_col='clamped_gradient',
                               bearing_change_raw_col='bearing_change', bearing_change_smooth_col='bearing_change_smoothed',
                               max_gradient=30.0):
    """
    Create a comprehensive 2x2 comparison plot showing raw vs smoothed data
    for both gradient and bearing changes.
    
    Args:
        df: DataFrame with gradient and bearing data
        title_prefix: Optional prefix for plot titles (e.g., course name)
        gradient_raw_col: Column name for raw gradient data
        gradient_smooth_col: Column name for smoothed gradient data
        bearing_change_raw_col: Column name for raw bearing change data
        bearing_change_smooth_col: Column name for smoothed bearing change data
        max_gradient: Maximum gradient threshold for reference lines
    
    Returns:
        plotly Figure object
    """
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Raw Gradient', 'Raw Bearing Changes',
            'Smoothed Gradient', 'Smoothed Bearing Changes'
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.08
    )
    
    # Row 1: Raw data
    # Gradient (raw)
    if gradient_raw_col in df.columns:
        fig.add_trace(
            go.Scatter(y=df[gradient_raw_col], mode='lines', name='Raw Gradient', 
                      line=dict(color='red', width=0.5)),
            row=1, col=1
        )
    
    # Bearing change (raw)
    if bearing_change_raw_col in df.columns:
        fig.add_trace(
            go.Scatter(y=df[bearing_change_raw_col], mode='lines', name='Raw Bearing', 
                      line=dict(color='red', width=0.5)),
            row=1, col=2
        )
    
    # Row 2: Smoothed data
    # Gradient (smoothed)
    if gradient_smooth_col in df.columns:
        fig.add_trace(
            go.Scatter(y=df[gradient_smooth_col], mode='lines', name='Smoothed Gradient', 
                      line=dict(color='blue')),
            row=2, col=1
        )
    
    # Bearing change (smoothed)
    if bearing_change_smooth_col in df.columns:
        fig.add_trace(
            go.Scatter(y=df[bearing_change_smooth_col], mode='lines', name='Smoothed Bearing', 
                      line=dict(color='blue')),
            row=2, col=2
        )
    
    # Add threshold lines for gradient
    for row in [1, 2]:
        fig.add_hline(y=max_gradient, line_dash='dash', line_color='orange', row=row, col=1)
        fig.add_hline(y=-max_gradient, line_dash='dash', line_color='orange', row=row, col=1)
    
    # Add threshold lines for bearing changes
    for row in [1, 2]:
        fig.add_hline(y=90, line_dash='dash', line_color='orange', row=row, col=2)
        fig.add_hline(y=-90, line_dash='dash', line_color='orange', row=row, col=2)
        fig.add_hline(y=120, line_dash='dash', line_color='red', row=row, col=2)
        fig.add_hline(y=-120, line_dash='dash', line_color='red', row=row, col=2)
    
    # Update axes labels
    fig.update_xaxes(title_text='Point Index', row=2, col=1)
    fig.update_xaxes(title_text='Point Index', row=2, col=2)
    fig.update_yaxes(title_text='Gradient (%)', row=1, col=1)
    fig.update_yaxes(title_text='Gradient (%)', row=2, col=1)
    fig.update_yaxes(title_text='Bearing Change (°)', row=1, col=2)
    fig.update_yaxes(title_text='Bearing Change (°)', row=2, col=2)
    
    # Layout
    title = f'{title_prefix} Raw vs Smoothed: Gradient & Bearing' if title_prefix else 'Raw vs Smoothed: Gradient & Bearing'
    fig.update_layout(
        height=700,
        width=1200,
        title_text=title,
        showlegend=False
    )
    
    return fig


def plot_smoothing_histograms(df, title_prefix="",
                              gradient_raw_col='raw_gradient', gradient_smooth_col='clamped_gradient',
                              bearing_change_raw_col='bearing_change', bearing_change_smooth_col='bearing_change_smoothed'):
    """
    Create histogram comparison plots for gradient and bearing changes.
    
    Args:
        df: DataFrame with gradient and bearing data
        title_prefix: Optional prefix for plot titles
        gradient_raw_col: Column name for raw gradient data
        gradient_smooth_col: Column name for smoothed gradient data
        bearing_change_raw_col: Column name for raw bearing change data
        bearing_change_smooth_col: Column name for smoothed bearing change data
    
    Returns:
        plotly Figure object
    """
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Gradient Distribution', 'Bearing Change Distribution'],
        horizontal_spacing=0.1
    )
    
    # Gradient histograms
    if gradient_raw_col in df.columns:
        fig.add_trace(
            go.Histogram(x=df[gradient_raw_col].dropna(), name='Raw Gradient', 
                        opacity=0.5, nbinsx=50, marker_color='red'),
            row=1, col=1
        )
    if gradient_smooth_col in df.columns:
        fig.add_trace(
            go.Histogram(x=df[gradient_smooth_col].dropna(), name='Smoothed Gradient', 
                        opacity=0.5, nbinsx=50, marker_color='blue'),
            row=1, col=1
        )
    
    # Bearing change histograms
    if bearing_change_raw_col in df.columns:
        fig.add_trace(
            go.Histogram(x=df[bearing_change_raw_col].dropna(), name='Raw Bearing', 
                        opacity=0.5, nbinsx=100, marker_color='red'),
            row=1, col=2
        )
    if bearing_change_smooth_col in df.columns:
        fig.add_trace(
            go.Histogram(x=df[bearing_change_smooth_col].dropna(), name='Smoothed Bearing', 
                        opacity=0.5, nbinsx=100, marker_color='blue'),
            row=1, col=2
        )
    
    # Add threshold lines
    fig.add_vline(x=90, line_dash='dash', line_color='red', row=1, col=2)
    fig.add_vline(x=-90, line_dash='dash', line_color='red', row=1, col=2)
    
    # Update layout
    fig.update_layout(
        barmode='overlay',
        height=400,
        width=1200,
        title_text=f'{title_prefix} Distribution Comparison' if title_prefix else 'Distribution Comparison',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
    )
    fig.update_xaxes(title_text='Gradient (%)', row=1, col=1)
    fig.update_xaxes(title_text='Bearing Change (°)', row=1, col=2)
    fig.update_yaxes(title_text='Count', row=1, col=1)
    fig.update_yaxes(title_text='Count', row=1, col=2)
    
    return fig


def print_smoothing_stats(df, 
                          gradient_raw_col='raw_gradient', gradient_smooth_col='clamped_gradient',
                          bearing_change_raw_col='bearing_change', bearing_change_smooth_col='bearing_change_smoothed',
                          max_gradient=30.0):
    """
    Print summary statistics comparing raw vs smoothed data for gradient and bearing.
    
    Args:
        df: DataFrame with gradient and bearing data
        gradient_raw_col: Column name for raw gradient data
        gradient_smooth_col: Column name for smoothed gradient data
        bearing_change_raw_col: Column name for raw bearing change data
        bearing_change_smooth_col: Column name for smoothed bearing change data
        max_gradient: Gradient threshold for counting extreme values
    """
    print("=" * 70)
    print("SMOOTHING STATISTICS COMPARISON")
    print("=" * 70)
    
    # Gradient stats
    print("\n--- GRADIENT ---")
    print(f"{'Metric':<25} {'Raw':>15} {'Smoothed':>15}")
    print("-" * 55)
    
    if gradient_raw_col in df.columns and gradient_smooth_col in df.columns:
        raw_g = df[gradient_raw_col]
        smooth_g = df[gradient_smooth_col]
        print(f"{'Min':<25} {raw_g.min():>15.1f}% {smooth_g.min():>15.1f}%")
        print(f"{'Max':<25} {raw_g.max():>15.1f}% {smooth_g.max():>15.1f}%")
        print(f"{'Mean':<25} {raw_g.mean():>15.2f}% {smooth_g.mean():>15.2f}%")
        print(f"{'Std Dev':<25} {raw_g.std():>15.2f}% {smooth_g.std():>15.2f}%")
        print(f"{'Points > {0}%'.format(max_gradient):<25} {(raw_g.abs() > max_gradient).sum():>15} {(smooth_g.abs() > max_gradient).sum():>15}")
    
    # Bearing stats
    print("\n--- BEARING CHANGES ---")
    print(f"{'Metric':<25} {'Raw':>15} {'Smoothed':>15}")
    print("-" * 55)
    
    if bearing_change_raw_col in df.columns and bearing_change_smooth_col in df.columns:
        raw_b = df[bearing_change_raw_col]
        smooth_b = df[bearing_change_smooth_col]
        print(f"{'Min':<25} {raw_b.min():>15.1f}° {smooth_b.min():>15.1f}°")
        print(f"{'Max':<25} {raw_b.max():>15.1f}° {smooth_b.max():>15.1f}°")
        print(f"{'Mean':<25} {raw_b.mean():>15.2f}° {smooth_b.mean():>15.2f}°")
        print(f"{'Std Dev':<25} {raw_b.std():>15.2f}° {smooth_b.std():>15.2f}°")
        print(f"{'Sharp turns (>90°)':<25} {(raw_b.abs() > 90).sum():>15} {(smooth_b.abs() > 90).sum():>15}")
        print(f"{'Extreme turns (>120°)':<25} {(raw_b.abs() > 120).sum():>15} {(smooth_b.abs() > 120).sum():>15}")
    
    print("=" * 70)

## 1. Load Raw GPS Data

Choose your data source: database sample course, CSV file, or Strava API

In [3]:
# Option 1: Load from database
db = CourseDatabase()
courses = db.conn.execute(
    "SELECT course_id, course_name, total_distance_km, total_elevation_gain_m, course_type FROM courses"
).fetchall()

print("Available courses:")
for i, c in enumerate(courses):
    print(f"  {i}: {c['course_name']} ({c['course_type']}) - {c['total_distance_km']:.1f}km, {c['total_elevation_gain_m']:.0f}m")

Available courses:
  0: Test Course for Integration (test_course) - 1.6km, 71m
  1: Pedal till ya puke (strava_segment) - 11.2km, 136m
  2: NF 4832 Climb (strava_segment) - 9.8km, 326m
  3: Flat Training Loop (sample_course) - 10.8km, 86m
  4: Mauna Kea Summit (sample_course) - 71.5km, 4387m


## 2. Raw Data Exploration

In [4]:
# Select a course (change index as needed)
COURSE_INDEX = 1  # <-- CHANGE THIS to select different course

selected_course = courses[COURSE_INDEX]
course_data = db.get_course(selected_course['course_id'])

# Get raw GPS data (before processing)
raw_df = pd.DataFrame(course_data['raw_gps_data'])

# Normalize column names: GPX files use 'ele', we standardize to 'altitude'
if 'ele' in raw_df.columns and 'altitude' not in raw_df.columns:
    raw_df['altitude'] = raw_df['ele']

print(f"Loaded: {selected_course['course_name']}")
print(f"Points: {len(raw_df)}")
print(f"Columns: {list(raw_df.columns)}")

# Combined: Map (row 1) + Altitude profile (row 2)
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Prepare map center
center_lat = raw_df['lat'].median() if 'lat' in raw_df.columns else 0
center_lon = raw_df['lon'].median() if 'lon' in raw_df.columns else 0

fig = make_subplots(rows=2, cols=1, specs=[[{'type': 'mapbox'}],[{'type': 'xy'}]],
                    subplot_titles=[f"Raw GPS Data - {selected_course['course_name']}", 'Raw Altitude Profile'])

# Map trace (row 1)
fig.add_trace(
    go.Scattermapbox(
        lat=raw_df['lat'],
        lon=raw_df['lon'],
        mode='markers',
        marker=dict(size=6, color=raw_df['altitude'], colorscale='Viridis', colorbar=dict(title='Elevation (m)')),
        hoverinfo='text',
        text=raw_df['altitude']
    ),
    row=1, col=1
)

# Altitude profile (row 2)
fig.add_trace(
    go.Scatter(y=raw_df['altitude'], mode='lines', name='Altitude', line=dict(color='blue')),
    row=2, col=1
)

# Layout adjustments
fig.update_layout(
    mapbox_style='open-street-map',
    mapbox=dict(center=dict(lat=center_lat, lon=center_lon), zoom=12),
    height=800,
    showlegend=False,
    title_text=f"Course: {selected_course['course_name']}"
)

fig.update_xaxes(title_text='Point Index', row=2, col=1)
fig.update_yaxes(title_text='Altitude (m)', row=2, col=1)
fig.show()

Loaded: Pedal till ya puke
Points: 984
Columns: ['lat', 'lon', 'ele', 'altitude']


/var/folders/4t/ph7592kn339630_gjktykq2h0000gn/T/ipykernel_5505/11829512.py:31: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [5]:
# Calculate raw gradient (without smoothing)
raw_df['delta_lat'] = raw_df['lat'].diff()
raw_df['delta_lon'] = raw_df['lon'].diff()
raw_df['distance'] = ((raw_df['delta_lat']**2 + raw_df['delta_lon']**2)**0.5) * 111139
raw_df['delta_altitude'] = raw_df['altitude'].diff()
raw_df['raw_gradient'] = (raw_df['delta_altitude'] / raw_df['distance']) * 100
raw_df['raw_gradient'] = raw_df['raw_gradient'].replace([np.inf, -np.inf], np.nan)

print(f"Raw gradient range: {raw_df['raw_gradient'].min():.1f}% to {raw_df['raw_gradient'].max():.1f}%")
print(f"Points with extreme gradient (>30%): {(raw_df['raw_gradient'].abs() > 30).sum()}")

Raw gradient range: -40.2% to 48.3%
Points with extreme gradient (>30%): 6


In [6]:
#Create a histogram with raw distance values
fig = px.histogram(raw_df, x='distance', nbins=50, title='Histogram of Raw Distance Between Points', labels={'distance': 'Distance (m)'})
fig.update_layout(bargap=0.1)
fig.show()

In [7]:
# Visualize raw gradient - notice the noise!
fig = px.line(raw_df, y='raw_gradient', title='Raw Gradient (Unsmoothed) - Notice the GPS Noise')
fig.update_layout(xaxis_title='Point Index', yaxis_title='Gradient (%)')
fig.add_hline(y=30, line_dash='dash', line_color='red', annotation_text='30% threshold')
fig.add_hline(y=-30, line_dash='dash', line_color='red')
fig.show()

In [8]:
# Calculate bearing (direction of travel) between consecutive points
def calculate_bearing(lat1, lon1, lat2, lon2):
    """Calculate bearing in degrees (0-360) from point 1 to point 2"""
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    delta_lon_rad = np.radians(lon2 - lon1)
    
    x = np.sin(delta_lon_rad) * np.cos(lat2_rad)
    y = np.cos(lat1_rad) * np.sin(lat2_rad) - np.sin(lat1_rad) * np.cos(lat2_rad) * np.cos(delta_lon_rad)
    
    bearing_rad = np.arctan2(x, y)
    bearing_deg = np.degrees(bearing_rad)
    
    # Normalize to 0-360
    return (bearing_deg + 360) % 360

# Calculate bearing for each segment (N-1 values)
bearings = calculate_bearing(
    raw_df['lat'].iloc[:-1].values,
    raw_df['lon'].iloc[:-1].values,
    raw_df['lat'].iloc[1:].values,
    raw_df['lon'].iloc[1:].values
)

# Pad with NaN at the end to match dataframe length
raw_df['bearing'] = np.concatenate([bearings, [np.nan]])

# Calculate bearing change (handle wrap-around at 0/360)
def bearing_change(bearing1, bearing2):
    """Calculate the smallest angle between two bearings"""
    diff = bearing2 - bearing1
    # Normalize to -180 to 180
    diff = np.where(diff > 180, diff - 360, diff)
    diff = np.where(diff < -180, diff + 360, diff)
    return diff

# Calculate bearing changes (N-2 values for valid bearings)
bearing_changes = bearing_change(
    raw_df['bearing'].iloc[:-1].values,
    raw_df['bearing'].iloc[1:].values
)

# Pad with NaN at the end to match dataframe length
raw_df['bearing_change'] = np.concatenate([bearing_changes, [np.nan]])

print(f"Bearing range: {raw_df['bearing'].min():.1f}° to {raw_df['bearing'].max():.1f}°")
print(f"Bearing change range: {raw_df['bearing_change'].min():.1f}° to {raw_df['bearing_change'].max():.1f}°")
print(f"Points with sharp turns (>90°): {(raw_df['bearing_change'].abs() > 90).sum()}")
print(f"Points with extreme turns (>120°): {(raw_df['bearing_change'].abs() > 120).sum()}")

Bearing range: 17.7° to 357.3°
Bearing change range: -113.4° to 160.9°
Points with sharp turns (>90°): 3
Points with extreme turns (>120°): 1


In [9]:
# Visualize bearing changes - notice GPS noise creates false turns!
fig = px.line(raw_df, y='bearing_change', title='Raw Bearing Changes - GPS Noise Creates False Turns')
fig.update_layout(xaxis_title='Point Index', yaxis_title='Bearing Change (degrees)')
fig.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='90° sharp turn')
fig.add_hline(y=-90, line_dash='dash', line_color='red')
fig.add_hline(y=120, line_dash='dash', line_color='orange', annotation_text='120° extreme turn')
fig.add_hline(y=-120, line_dash='dash', line_color='orange')
fig.show()

In [10]:
# Histogram of bearing changes
fig = px.histogram(raw_df, x='bearing_change', nbins=100, 
                   title='Distribution of Bearing Changes',
                   labels={'bearing_change': 'Bearing Change (degrees)'})
fig.add_vline(x=90, line_dash='dash', line_color='red')
fig.add_vline(x=-90, line_dash='dash', line_color='red')
fig.show()

## 3. Experiment with Smoothing Parameters

The Savitzky-Golay filter smooths altitude while preserving peaks and valleys. Key parameters:
- **window_length**: Number of points to include (must be odd)
- **polyorder**: Polynomial order for fitting (2 works well for altitude)

In [11]:
# Smoothing parameters - EXPERIMENT WITH THESE
WINDOW_LENGTH = 21  # Must be odd, try: 5, 11, 21, 31, 51
POLYORDER = 2       # Polynomial order, typically 2 or 3
MAX_GRADIENT = 30.0 # Maximum gradient in percent

# Apply Savitzky-Golay smoothing
if WINDOW_LENGTH >= 5 and len(raw_df) >= WINDOW_LENGTH:
    altitude_smoothed = savgol_filter(
        raw_df['altitude'].ffill().bfill(),
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )
else:
    altitude_smoothed = raw_df['altitude'].values

raw_df['altitude_smoothed'] = altitude_smoothed

In [12]:
# Compare raw vs smoothed altitude
fig = go.Figure()
fig.add_trace(go.Scatter(y=raw_df['altitude'], name='Raw Altitude', line=dict(color='lightgray')))
fig.add_trace(go.Scatter(y=raw_df['altitude_smoothed'], name=f'Smoothed (window={WINDOW_LENGTH})', line=dict(color='blue')))
fig.update_layout(
    title=f'Altitude Smoothing Comparison (window={WINDOW_LENGTH}, polyorder={POLYORDER})',
    xaxis_title='Point Index',
    yaxis_title='Altitude (m)'
)
fig.show()

In [13]:
# Calculate gradient from smoothed altitude
raw_df['delta_altitude_smooth'] = pd.Series(raw_df['altitude_smoothed']).diff()
raw_df['smooth_gradient'] = (raw_df['delta_altitude_smooth'] / raw_df['distance']) * 100
raw_df['smooth_gradient'] = raw_df['smooth_gradient'].replace([np.inf, -np.inf], np.nan)

# Apply clamping
raw_df['clamped_gradient'] = raw_df['smooth_gradient'].clip(lower=-MAX_GRADIENT, upper=MAX_GRADIENT)

print(f"Smoothed gradient range: {raw_df['smooth_gradient'].min():.1f}% to {raw_df['smooth_gradient'].max():.1f}%")
print(f"After clamping (±{MAX_GRADIENT}%): {raw_df['clamped_gradient'].min():.1f}% to {raw_df['clamped_gradient'].max():.1f}%")

Smoothed gradient range: -10.7% to 31.0%
After clamping (±30.0%): -10.7% to 30.0%


In [14]:
# Compare raw vs smoothed gradient
fig = make_subplots(rows=2, cols=1, subplot_titles=['Raw Gradient', 'Smoothed + Clamped Gradient'])

fig.add_trace(
    go.Scatter(y=raw_df['raw_gradient'], name='Raw', line=dict(color='red', width=0.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(y=raw_df['clamped_gradient'], name='Smoothed+Clamped', line=dict(color='blue')),
    row=2, col=1
)

fig.update_layout(height=600, title_text=f'Gradient Comparison (window={WINDOW_LENGTH}, clamp=±{MAX_GRADIENT}%)')
fig.show()

In [15]:
# Apply smoothing to bearings (handling circular nature of angles)
# Convert bearings to unit vectors, smooth, then convert back

# Convert bearing to x,y components
bearing_x = np.cos(np.radians(raw_df['bearing']))
bearing_y = np.sin(np.radians(raw_df['bearing']))

# Smooth the components using the same window parameters as altitude
if WINDOW_LENGTH >= 5 and len(raw_df) >= WINDOW_LENGTH:
    bearing_x_smooth = savgol_filter(
        pd.Series(bearing_x).ffill().bfill(),
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )
    bearing_y_smooth = savgol_filter(
        pd.Series(bearing_y).ffill().bfill(),
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )
else:
    bearing_x_smooth = bearing_x
    bearing_y_smooth = bearing_y

# Convert back to bearing angles
raw_df['bearing_smoothed'] = np.degrees(np.arctan2(bearing_y_smooth, bearing_x_smooth)) % 360

# Calculate smoothed bearing changes
bearing_changes_smooth = bearing_change(
    raw_df['bearing_smoothed'].iloc[:-1].values,
    raw_df['bearing_smoothed'].iloc[1:].values
)
raw_df['bearing_change_smoothed'] = np.concatenate([bearing_changes_smooth, [np.nan]])

print(f"Smoothed bearing change range: {raw_df['bearing_change_smoothed'].min():.1f}° to {raw_df['bearing_change_smoothed'].max():.1f}°")
print(f"Smoothed sharp turns (>90°): {(raw_df['bearing_change_smoothed'].abs() > 90).sum()}")
print(f"Smoothed extreme turns (>120°): {(raw_df['bearing_change_smoothed'].abs() > 120).sum()}")

Smoothed bearing change range: -6.4° to 6.1°
Smoothed sharp turns (>90°): 0
Smoothed extreme turns (>120°): 0


In [16]:
# Compare raw vs smoothed bearing changes
fig = make_subplots(rows=2, cols=1, subplot_titles=['Raw Bearing Changes', 'Smoothed Bearing Changes'])

fig.add_trace(
    go.Scatter(y=raw_df['bearing_change'], name='Raw', line=dict(color='red', width=0.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(y=raw_df['bearing_change_smoothed'], name='Smoothed', line=dict(color='blue')),
    row=2, col=1
)

# Add threshold lines to both subplots
for row in [1, 2]:
    fig.add_hline(y=90, line_dash='dash', line_color='orange', row=row, col=1)
    fig.add_hline(y=-90, line_dash='dash', line_color='orange', row=row, col=1)
    fig.add_hline(y=120, line_dash='dash', line_color='red', row=row, col=1)
    fig.add_hline(y=-120, line_dash='dash', line_color='red', row=row, col=1)

fig.update_xaxes(title_text='Point Index', row=2, col=1)
fig.update_yaxes(title_text='Bearing Change (°)', row=1, col=1)
fig.update_yaxes(title_text='Bearing Change (°)', row=2, col=1)
fig.update_layout(height=600, title_text=f'Bearing Change Comparison (window={WINDOW_LENGTH}, polyorder={POLYORDER})', showlegend=False)
fig.show()

In [17]:
# Histogram comparison: raw vs smoothed bearing changes
fig = go.Figure()
fig.add_trace(go.Histogram(x=raw_df['bearing_change'].dropna(), name='Raw', opacity=0.5, nbinsx=100))
fig.add_trace(go.Histogram(x=raw_df['bearing_change_smoothed'].dropna(), name='Smoothed', opacity=0.5, nbinsx=100))
fig.update_layout(
    title='Bearing Change Distribution: Raw vs Smoothed',
    xaxis_title='Bearing Change (degrees)',
    yaxis_title='Count',
    barmode='overlay'
)
fig.add_vline(x=90, line_dash='dash', line_color='red')
fig.add_vline(x=-90, line_dash='dash', line_color='red')
fig.show()

In [18]:
# Use the reusable comparison functions - Combined view of all smoothing results
fig = plot_smoothing_comparison(raw_df, title_prefix=selected_course['course_name'])
fig.show()

# Show histograms
fig_hist = plot_smoothing_histograms(raw_df, title_prefix=selected_course['course_name'])
fig_hist.show()

# Print statistics
print_smoothing_stats(raw_df, max_gradient=MAX_GRADIENT)

SMOOTHING STATISTICS COMPARISON

--- GRADIENT ---
Metric                                Raw        Smoothed
-------------------------------------------------------
Min                                 -40.2%           -10.7%
Max                                  48.3%            30.0%
Mean                                 0.48%            0.68%
Std Dev                              5.27%            4.14%
Points > 30.0%                          6               0

--- BEARING CHANGES ---
Metric                                Raw        Smoothed
-------------------------------------------------------
Min                                -113.4°            -6.4°
Max                                 160.9°             6.1°
Mean                                 0.00°            0.00°
Std Dev                             14.56°            1.30°
Sharp turns (>90°)                      3               0
Extreme turns (>120°)                   1               0


## 4. Adaptive Window Size

The production code uses an adaptive window that targets ~50m of horizontal distance. This adapts to different GPS sampling densities.

In [19]:
# Calculate adaptive window size (as in production code)
avg_point_spacing = raw_df['distance'].median()
print(f"Median point spacing: {avg_point_spacing:.1f}m")

TARGET_DISTANCE = 50  # meters - EXPERIMENT WITH THIS

target_points = max(5, min(51, int(TARGET_DISTANCE / avg_point_spacing)))
adaptive_window = target_points if target_points % 2 == 1 else target_points + 1

print(f"Target distance for smoothing: {TARGET_DISTANCE}m")
print(f"Calculated adaptive window: {adaptive_window} points")
print(f"Actual smoothing distance: {adaptive_window * avg_point_spacing:.1f}m")

Median point spacing: 11.4m
Target distance for smoothing: 50m
Calculated adaptive window: 5 points
Actual smoothing distance: 56.9m


In [20]:
# Apply adaptive smoothing
altitude_adaptive = savgol_filter(
    raw_df['altitude'].ffill().bfill(),
    window_length=adaptive_window,
    polyorder=2
)

raw_df['altitude_adaptive'] = altitude_adaptive
raw_df['delta_altitude_adaptive'] = pd.Series(altitude_adaptive).diff()
raw_df['adaptive_gradient'] = (raw_df['delta_altitude_adaptive'] / raw_df['distance']) * 100
raw_df['adaptive_gradient'] = raw_df['adaptive_gradient'].replace([np.inf, -np.inf], np.nan)
raw_df['adaptive_gradient_clamped'] = raw_df['adaptive_gradient'].clip(lower=-30, upper=30)

print(f"Adaptive gradient range: {raw_df['adaptive_gradient_clamped'].min():.1f}% to {raw_df['adaptive_gradient_clamped'].max():.1f}%")

Adaptive gradient range: -30.0% to 30.0%


## 5. Gradient Distribution Analysis

In [21]:
# Histogram comparison
fig = go.Figure()
fig.add_trace(go.Histogram(x=raw_df['raw_gradient'].dropna(), name='Raw', opacity=0.5, nbinsx=50))
fig.add_trace(go.Histogram(x=raw_df['adaptive_gradient_clamped'].dropna(), name='Smoothed+Clamped', opacity=0.5, nbinsx=50))
fig.update_layout(
    title='Gradient Distribution: Raw vs Smoothed',
    xaxis_title='Gradient (%)',
    yaxis_title='Count',
    barmode='overlay'
)
fig.show()

In [22]:
# Summary statistics
print("Gradient Statistics Comparison:")
print("="*50)
print(f"{'Metric':<20} {'Raw':>12} {'Smoothed':>12}")
print("-"*50)
print(f"{'Min':<20} {raw_df['raw_gradient'].min():>12.1f}% {raw_df['adaptive_gradient_clamped'].min():>12.1f}%")
print(f"{'Max':<20} {raw_df['raw_gradient'].max():>12.1f}% {raw_df['adaptive_gradient_clamped'].max():>12.1f}%")
print(f"{'Mean':<20} {raw_df['raw_gradient'].mean():>12.2f}% {raw_df['adaptive_gradient_clamped'].mean():>12.2f}%")
print(f"{'Std Dev':<20} {raw_df['raw_gradient'].std():>12.2f}% {raw_df['adaptive_gradient_clamped'].std():>12.2f}%")
print(f"{'Points > 30%':<20} {(raw_df['raw_gradient'].abs() > 30).sum():>12} {(raw_df['adaptive_gradient_clamped'].abs() > 30).sum():>12}")

Gradient Statistics Comparison:
Metric                        Raw     Smoothed
--------------------------------------------------
Min                         -40.2%        -30.0%
Max                          48.3%         30.0%
Mean                         0.48%         0.50%
Std Dev                      5.27%         4.89%
Points > 30%                    6            0


## 6. Process with Production Course Class

Finally, let's verify the production `Course` class produces the same results.

In [24]:
# Create Course object (applies production smoothing)
course = Course(pd.DataFrame(course_data['raw_gps_data']))

print(f"Production Course object:")
print(f"  Points: {len(course.df)}")
print(f"  Gradient range: {course.df['gradient'].min():.1f}% to {course.df['gradient'].max():.1f}%")
print(f"  Total distance: {course.df['cumulative_distance'].iloc[-1]/1000:.2f} km")
print(f"  Total elevation gain: {course.df['height_gain'].sum():.0f} m")

running process segment
Production Course object:
  Points: 984
  Gradient range: -5.9% to 16.8%
  Total distance: 11.23 km
  Total elevation gain: 136 m


In [26]:
course = Course(
    pd.DataFrame(course_data['raw_gps_data']),
    smoothing_target_distance=100.0,  # meters
    smoothing_min_window=51,           # points
    smoothing_max_window=101,          # points  
    smoothing_polyorder=2,
    max_gradient=25.0                  # percent
)
# Plot smoothing comparison using the Course object's method
fig = course.plot_smoothing_comparison(title_prefix=selected_course['course_name'])
fig.show()
course.print_smoothing_stats()

running process segment


SMOOTHING STATISTICS COMPARISON

--- GRADIENT ---
Metric                                Raw        Smoothed
-------------------------------------------------------
Min                                 -40.2%            -5.9%
Max                                  48.3%            16.8%
Mean                                 0.48%            0.68%
Std Dev                              5.27%            3.31%
Points > 30.0%                          6               0

--- BEARING CHANGES ---
Metric                                Raw        Smoothed
-------------------------------------------------------
Min                                -113.4°            -2.6°
Max                                 160.9°             2.7°
Mean                                -0.09°            0.00°
Std Dev                             14.83°            0.54°
Sharp turns (>90°)                      3               0
Extreme turns (>120°)                   1               0
